# h4lat — The 16-cell honeycomb group

**Clebsch-Gordan coefficients and lattice operators for the symmetry group W(F₄) of the 16-cell honeycomb lattice**

The 16-cell honeycomb is the hypercubic lattice ℤ⁴ together with its copy shifted by (½, ½, ½, ½). Its point group,
W(F₄), has 1152 elements and 25 irreducible representations (irreps), and contains the hypercubic group H(4).
S. Katz and D. Nogradi ([arXiv:2611.21543](https://arxiv.org/abs/2611.21543)) showed that, thanks to the larger
symmetry, more of the operators for the moments of the parton distributions renormalise as in the continuum and can be
measured in a nucleon at rest. h4lat supports this group with the argument `group="honeycomb"`; the irrep labels
ϱ⁽ᵏ⁾ₗ, written `(k, l)` in the code, are those of the paper. The theory is on the wiki page
[The 16-cell honeycomb group](https://github.com/Tag-E/h4lat/wiki/The-16-cell-honeycomb-group).

Run the cells top to bottom.

---
## 1. The group

`get_group("honeycomb")` returns the object that describes the group: its irreps, character table and class sizes.
The character table was computed with [GAP](https://www.gap-system.org) and is checked by the tests of h4lat against
the definitions of the irreps in the paper.

In [1]:
from h4lat import get_group

honeycomb = get_group("honeycomb")
h4 = get_group("H4")  # the default group, also get_group()
print(honeycomb)
print(h4)
print()
print(f"{'Label':<9} {'Dim':>4}  LaTeX name")
print("-" * 40)
for label in honeycomb.rep_label_list:
    print(f"{str(label):<9} {honeycomb.irrep_dim[label]:>4}  ${honeycomb.irrep_texname[label]}$")
print()
print("Σ dim² =", sum(d * d for d in honeycomb.rep_dim_list), "= order", honeycomb.order)

LatticeGroup('honeycomb': W(F₄), order 1152, 25 irreps)
LatticeGroup('H4': H(4), order 384, 20 irreps)

Label      Dim  LaTeX name
----------------------------------------
(1, 1)       1  $\varrho^{\left(1\right)}_1$
(1, 2)       1  $\varrho^{\left(1\right)}_2$
(1, 3)       1  $\varrho^{\left(1\right)}_3$
(1, 4)       1  $\varrho^{\left(1\right)}_4$
(2, 1)       2  $\varrho^{\left(2\right)}_1$
(2, 2)       2  $\varrho^{\left(2\right)}_2$
(2, 3)       2  $\varrho^{\left(2\right)}_3$
(2, 4)       2  $\varrho^{\left(2\right)}_4$
(4, 1)       4  $\varrho^{\left(4\right)}_1$
(4, 2)       4  $\varrho^{\left(4\right)}_2$
(4, 3)       4  $\varrho^{\left(4\right)}_3$
(4, 4)       4  $\varrho^{\left(4\right)}_4$
(4, 5)       4  $\varrho^{\left(4\right)}_5$
(6, 1)       6  $\varrho^{\left(6\right)}_1$
(6, 2)       6  $\varrho^{\left(6\right)}_2$
(8, 1)       8  $\varrho^{\left(8\right)}_1$
(8, 2)       8  $\varrho^{\left(8\right)}_2$
(8, 3)       8  $\varrho^{\left(8\right)}_3$
(8, 4)       8  $\

---
## 2. The elements

h4lat builds the 1152 elements from its own H(4) matrices and the triality rotation A of the paper (eq. 3.2): element
`384*j + i` is the `i`-th element of H(4) times Aʲ. The matrices of all the elements in every irrep are bundled with the
package and are held by the attribute `group_mat` of a `cg_calc` object.

In [2]:
import numpy as np

from h4lat import cg_calc
from h4lat.honeycomb import GENERATORS, TRIALITY

cg = cg_calc((4, 1), (4, 1), group="honeycomb", verbose=False)  # a bundled product: nothing is computed
vector = cg.group_mat[honeycomb.irrep_index[(4, 1)]]            # the 1152 matrices of the vector irrep (4,1)
print("shape:", vector.shape)
print("element 384 is the triality rotation A:", np.array_equal(vector[384], TRIALITY))
print("A^3 = 1:", np.allclose(np.linalg.matrix_power(TRIALITY, 3), np.eye(4)))

h4_vector = cg_calc((4, 1), (4, 1), verbose=False).group_mat[h4.irrep_index[(4, 1)]]
print("elements 0-383 are those of H(4):", np.array_equal(vector[:384], h4_vector))

keys = {tuple(np.rint(2 * g).astype(int).ravel()) for g in vector}
print("the generators g1, g2 of the paper are elements:", all(tuple(np.rint(2 * g).astype(int).ravel()) in keys for g in GENERATORS))

shape: (1152, 4, 4)
element 384 is the triality rotation A: True
A^3 = 1: True


elements 0-383 are those of H(4): True
the generators g1, g2 of the paper are elements: True


---
## 3. Tensor products

The decomposition of a tensor product works as for H(4), with `group="honeycomb"`. The product of two vectors already
shows the difference: the symmetric traceless tensors stay a single irrep, (9,1), while H(4) splits them into (3,1)
and (6,3).

In [3]:
from h4lat import get_multiplicities, latex_print_multiplicities


def decomposition(*irreps, group=None):
    """The non-zero multiplicities of a tensor product, as a dict {irrep label: multiplicity}."""
    labels = get_group(group).rep_label_list
    return {labels[i]: m for i, m in enumerate(get_multiplicities(*irreps, group=group)) if m}


for product in [((4, 1), (4, 1)), ((4, 1), (4, 1), (4, 1)), ((4, 4), (4, 1), (4, 1))]:
    print(" ⊗ ".join(map(str, product)))
    print("   H(4)     :", decomposition(*product))
    print("   honeycomb:", decomposition(*product, group="honeycomb"))

(4, 1) ⊗ (4, 1)
   H(4)     : {(1, 1): 1, (3, 1): 1, (6, 1): 1, (6, 3): 1}
   honeycomb: {(1, 1): 1, (6, 1): 1, (9, 1): 1}
(4, 1) ⊗ (4, 1) ⊗ (4, 1)
   H(4)     : {(4, 1): 4, (4, 2): 1, (4, 4): 1, (8, 1): 3, (8, 2): 2}
   honeycomb: {(4, 1): 3, (4, 4): 1, (8, 1): 1, (8, 3): 1, (16, 1): 2}
(4, 4) ⊗ (4, 1) ⊗ (4, 1)
   H(4)     : {(4, 1): 1, (4, 3): 1, (4, 4): 4, (8, 1): 2, (8, 2): 3}
   honeycomb: {(4, 1): 1, (4, 4): 3, (8, 2): 1, (8, 4): 1, (16, 1): 2}


In [4]:
latex_print_multiplicities((4, 1), (4, 1), group="honeycomb", display_eq=True, return_str=False)
latex_print_multiplicities((4, 1), (4, 1), display_eq=True, return_str=False)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---
## 4. How the irreps of W(F₄) split into those of H(4)

Restricting an irrep of W(F₄) to the elements of H(4) (the first 384) gives a representation of H(4), which the
characters decompose into irreps of H(4) (eq. 5.4 of the paper). Several pairs of H(4) irreps fuse into one irrep of
the honeycomb group.

In [5]:
cg_h4 = cg_calc((4, 1), (4, 1), verbose=False)
h4_characters = {label: np.trace(m, axis1=1, axis2=2) for label, m in zip(h4.rep_label_list, cg_h4.group_mat)}

for label, matrices in zip(honeycomb.rep_label_list, cg.group_mat):
    chi = np.trace(matrices[:384], axis1=1, axis2=2)  # the character on the elements of H(4)
    parts = {h4_label: round(chi @ h4_chi / 384) for h4_label, h4_chi in h4_characters.items()}
    print(f"{str(label):<8} → " + " ⊕ ".join(f"{m if m > 1 else ''}{h4_label}" for h4_label, m in parts.items() if m))

(1, 1)   → (1, 1)
(1, 2)   → (1, 2)
(1, 3)   → (1, 3)
(1, 4)   → (1, 4)
(2, 1)   → (2, 1)
(2, 2)   → (2, 2)
(2, 3)   → (1, 1) ⊕ (1, 2)
(2, 4)   → (1, 3) ⊕ (1, 4)
(4, 1)   → (4, 1)
(4, 2)   → (4, 2)
(4, 3)   → (4, 3)
(4, 4)   → (4, 4)
(4, 5)   → (2, 1) ⊕ (2, 2)
(6, 1)   → (6, 1)
(6, 2)   → (6, 2)
(8, 1)   → (8, 1)
(8, 2)   → (8, 2)
(8, 3)   → (4, 1) ⊕ (4, 2)
(8, 4)   → (4, 3) ⊕ (4, 4)
(9, 1)   → (3, 1) ⊕ (6, 3)
(9, 2)   → (3, 2) ⊕ (6, 3)
(9, 3)   → (3, 3) ⊕ (6, 4)
(9, 4)   → (3, 4) ⊕ (6, 4)
(12, 1)  → (6, 1) ⊕ (6, 2)
(16, 1)  → (8, 1) ⊕ (8, 2)


---
## 5. Clebsch-Gordan coefficients

The CG coefficients of the honeycomb group are bundled for the same products as for H(4). The block of the irrep (9,1)
in (4,1) ⊗ (4,1) spans the same space as the blocks of (3,1) and (6,3) of H(4).

In [6]:
block_9 = cg.cg_dict[honeycomb.irrep_index[(9, 1)]][0]  # shape (16, 9): rows = basis of (4,1)⊗(4,1)
print("shape of the (9,1) block:", block_9.shape)

blocks_h4 = np.hstack([cg_h4.cg_dict[h4.irrep_index[label]][0] for label in [(3, 1), (6, 3)]])
rank_together = np.linalg.matrix_rank(np.hstack([block_9, blocks_h4]))
print("rank of (9,1) of W(F4):", np.linalg.matrix_rank(block_9))
print("rank of (3,1) ⊕ (6,3) of H(4):", np.linalg.matrix_rank(blocks_h4))
print("rank of both together:", rank_together, "→ the same 9-dimensional space")

shape of the (9,1) block: (16, 9)
rank of (9,1) of W(F4): 9
rank of (3,1) ⊕ (6,3) of H(4): 9
rank of both together: 9 → the same 9-dimensional space


---
## 6. The operator database

The bundled database of the honeycomb group holds the same 1360 operators (S, P, V, A, T with 0 to 3 derivatives),
sorted by its irreps, with the same ID ranges as the H(4) database. Every operator records its group.

In [7]:
from h4lat import get_OperatorDict, get_OperatorList

ops = get_OperatorList(group="honeycomb")
print(len(ops), "operators; the first one:", ops[0].id, ops[0].X, ops[0].irrep, ops[0].group)

d = get_OperatorDict(keys="V1", group="honeycomb")  # vector operators with one derivative
for (irrep, block), block_ops in d[(2, "V")].items():
    print(f"  irrep {irrep}, copy {block}: {len(block_ops)} operators, C = {block_ops[0].C}, {block_ops[0].symm}")

1360 operators; the first one: 1 S (1, 1) honeycomb
  irrep (1, 1), copy 1: 1 operators, C = 1, Symmetric
  irrep (6, 1), copy 1: 6 operators, C = 1, Antisymmetric
  irrep (9, 1), copy 1: 9 operators, C = 1, Symmetric


The kinematic factor K of an operator shows whether it can be measured in a nucleon at rest, p = (0, 0, 0, iE):
the operators of (9,1), which give the momentum fraction ⟨x⟩ (v₂), include some with K ≠ 0 at rest.

In [8]:
import sympy as sym

from h4lat.kinematic_data import p1, p2, p3

at_rest = {p1: 0, p2: 0, p3: 0}
for op in d[(2, "V")][(9, 1), 1]:
    print(f"operator {op.id:>3}:  K at rest = {sym.simplify(op.K.subs(at_rest))}")

operator  32:  K at rest = -I*m_N**2/(3*E(p))
operator  33:  K at rest = 0
operator  34:  K at rest = 0
operator  35:  K at rest = 0


operator  36:  K at rest = -I*m_N**2/(2*E(p))
operator  37:  K at rest = 0
operator  38:  K at rest = 0


operator  39:  K at rest = -I*m_N**2/E(p)
operator  40:  K at rest = 0


---
## 7. The channels measurable at rest

Katz and Nogradi (secs. 7–9) single out, for each moment, an irrep of the honeycomb group that renormalises as in the
continuum and can be measured at rest. The cell below checks, for each of them, that some operator of the irrep has a
non-zero kinematic factor at rest (with the nucleon polarised along z, the default projector), and shows for
comparison an irrep of the same operators that cannot be measured at rest. It computes the kinematic factors of about
a hundred operators, which takes a few seconds.
The explicit operators that the paper gives for these channels (its eqs. 7.20, 8.12 and 9.17) are returned by
`get_paper_operators("Katz:2026qmx")`: see [`paper_operators.ipynb`](paper_operators.ipynb).

In [9]:
def measurable_at_rest(X, nder, irrep):
    """Whether some operator of the irrep has a non-zero kinematic factor at rest (p = 0, spin along z)."""
    operators = get_OperatorDict(keys=f"{X}{nder}", group="honeycomb")
    n = nder + 1  # V and A have one Lorentz index more than derivatives
    blocks = [block_ops for (label, _), block_ops in operators[(n, X)].items() if label == irrep]
    return any(sym.simplify(op.K.subs(at_rest)) != 0 for block_ops in blocks for op in block_ops)


channels = [
    ("v₂ = ⟨x⟩", "V", 1, (9, 1)),
    ("v₃ = ⟨x²⟩", "V", 2, (8, 3)),
    ("      (not at rest)", "V", 2, (8, 1)),
    ("v₄ = ⟨x³⟩", "V", 3, (2, 3)),
    ("      (not at rest)", "V", 3, (2, 1)),
    ("a₁", "A", 1, (9, 4)),
    ("a₂", "A", 2, (8, 2)),
    ("a₂", "A", 2, (8, 4)),
    ("d₂", "A", 2, (16, 1)),
    ("d₃", "A", 3, (9, 3)),
]
for name, X, nder, irrep in channels:
    print(f"{name:<20} {X}{nder}  irrep {str(irrep):<8}  measurable at rest: {measurable_at_rest(X, nder, irrep)}")

v₂ = ⟨x⟩             V1  irrep (9, 1)    measurable at rest: True


v₃ = ⟨x²⟩            V2  irrep (8, 3)    measurable at rest: True


      (not at rest)  V2  irrep (8, 1)    measurable at rest: False


v₄ = ⟨x³⟩            V3  irrep (2, 3)    measurable at rest: True


      (not at rest)  V3  irrep (2, 1)    measurable at rest: False


a₁                   A1  irrep (9, 4)    measurable at rest: True


a₂                   A2  irrep (8, 2)    measurable at rest: True


a₂                   A2  irrep (8, 4)    measurable at rest: True


d₂                   A2  irrep (16, 1)   measurable at rest: True


d₃                   A3  irrep (9, 3)    measurable at rest: True


---
## 8. Saving operators

Operators of the honeycomb group keep their group when saved: the file names written by `Operator.save` end in
`_honeycomb`, and HDF5 files store the attribute `symmetry_group`. Files without a group are read as H(4) operators.

In [10]:
import tempfile
from pathlib import Path

from h4lat import Operator_from_file

op = d[(2, "V")][(9, 1), 1][0]
with tempfile.TemporaryDirectory() as folder:
    op.save(folder)
    (path,) = Path(folder).iterdir()
    print(path.name)
    print(Operator_from_file(path).group, Operator_from_file(path).irrep)

operator_32_2_V_9_1_1_1_honeycomb.npy
honeycomb (9, 1)


---
## References

- S. Katz and D. Nogradi, *Hadron structure on the 16-cell honeycomb*, [arXiv:2611.21543](https://arxiv.org/abs/2611.21543).
- S. Katz and D. Nogradi, *QCD on the 16-cell honeycomb*, Phys. Rev. D **114**, 054504 (2026), [arXiv:2512.10604](https://arxiv.org/abs/2512.10604).
- The GAP Group, *GAP – Groups, Algorithms, and Programming*, [www.gap-system.org](https://www.gap-system.org).